# Paper §5.3 — Routing-and-integration synthesis

Narrative-only notebook that loads the §5.1 + §5.2 outputs and lays out the routing-and-integration framework the paper uses to synthesize them. No new compute.

**Spec source-of-truth.** `docs/paper/emnlp_outline_ko.md` — §5.3 (Routing-and-integration).

This notebook drives the heavy inference stages by `subprocess`-invoking
the existing sharded drivers in `scripts/`. The `RUN_INFERENCE = False`
toggle below lets a reviewer read the entire pipeline without GPU
access. Full reproduction targets the **8 × H200** cluster and uses
`--gpus 0,1,2,3,4,5,6,7` end-to-end.


## 1 · Setup — paths, GPU sharding, subprocess helper

In [ ]:
from __future__ import annotations
import json
import os
import shlex
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_main_worktree() -> Path:
    # Gitignored artifacts (inputs/, outputs/, docs/insights/_data/) live in
    # the main worktree even when this notebook runs from a linked worktree.
    common = subprocess.check_output(
        ["git", "rev-parse", "--git-common-dir"], cwd=Path.cwd(), text=True
    ).strip()
    return Path(common).resolve().parent


def find_worktree_root() -> Path:
    # Current worktree's working tree top (== main when not in a linked worktree).
    return Path(subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], cwd=Path.cwd(), text=True
    ).strip()).resolve()


MAIN     = find_main_worktree()
WORKTREE = find_worktree_root()

# Outputs live under MAIN (gitignored); figures land under WORKTREE so they ride the active branch.
# Scripts + configs come from WORKTREE so the active branch's edits are used
# (subprocess invocations would otherwise hit the stale main-worktree copies
# until the active PR merges).
SCRIPTS    = WORKTREE / "scripts"
CONFIGS    = WORKTREE / "configs"
DATA_DIR   = MAIN / "docs" / "insights" / "_data"

# §5.2 reads predictions.jsonl. Under H1 (raw-number prompt + eps=0 DF)
# point at the paper2 snapshot. Legacy timestamped run dirs preserved as
# fallback for audit-trail / direct rerun-without-H1 reproduction.
H1_PRED_ROOT = MAIN / "outputs" / "paper2" / "cross_model_cross_dataset" / "predictions"
H1_PRED_ROOTS = {
    "plotqa":         H1_PRED_ROOT / "plotqa"         / "llava-onevision-qwen2-7b-ov",
    "infographicvqa": H1_PRED_ROOT / "infographicvqa" / "llava-onevision-qwen2-7b-ov",
    "chartqa":        H1_PRED_ROOT / "chartqa"        / "llava-onevision-qwen2-7b-ov",
    "mathvista":      H1_PRED_ROOT / "mathvista"      / "llava-onevision-qwen2-7b-ov",
    "tallyqa":        H1_PRED_ROOT / "tallyqa"        / "llava-onevision-qwen2-7b-ov",
}
LEGACY_PRED_ROOTS = {
    "plotqa":         MAIN / "outputs" / "experiment_e7_plotqa_full"         / "llava-onevision-qwen2-7b-ov" / "20260502-132624",
    "infographicvqa": MAIN / "outputs" / "experiment_e7_infographicvqa_full" / "llava-onevision-qwen2-7b-ov" / "20260502-152105",
    "chartqa":        MAIN / "outputs" / "experiment_e5e_chartqa_full"       / "llava-onevision-qwen2-7b-ov" / "20260502-211028",
    "mathvista":      MAIN / "outputs" / "experiment_e5e_mathvista_full"     / "llava-onevision-qwen2-7b-ov" / "20260502-212440",
    "tallyqa":        MAIN / "outputs" / "experiment_e5e_tallyqa_full"       / "llava-onevision-qwen2-7b-ov" / "20260502-083926",
}


def legacy_pred(ds_tag: str) -> Path:
    return LEGACY_PRED_ROOTS[ds_tag] / "predictions.jsonl"


def h1_pred(ds_tag: str) -> Path:
    return H1_PRED_ROOTS[ds_tag] / "predictions.jsonl"

# §5.2 e6_steering input root selection (same toggle as §5.1 above):
#   - RUN_INFERENCE=False: read pre-existing sweep dirs from legacy
#     `outputs/e6_steering/` (the aggregators have always filtered by
#     subdirectory name, so the legacy tree won't pollute results even
#     when read-only).
#   - RUN_INFERENCE=True: write new calibrate / sweep dirs to an isolated
#     tree so this run doesn't commingle with the legacy pool.
E6_ROOT_LEGACY = MAIN / "outputs" / "e6_steering"
E6_ROOT_FRESH  = MAIN / "outputs" / "paper2" / "section_5_e6_steering"

# §5.1 attention input root selection:
#   - RUN_INFERENCE=False: read pre-existing bbox-with runs from
#     legacy outputs/attention_analysis/ (the docstring header in
#     `extract_attention_mass.py` confirms each model has multiple
#     timestamped runs carrying `image_anchor_digit`; the analyzer
#     auto-skips bbox-less records via field-presence check).
#   - RUN_INFERENCE=True: write new runs to a fresh isolated tree so
#     they do not commingle with the legacy pool.
ATT_ROOT_LEGACY = MAIN / "outputs" / "attention_analysis"
# `section_5_attention` is the n=400 root (n=400 spec was the first run).
# `section_5_attention_n1000` is the n=1000 extension. After n=1000 completes,
# n=400 is to be retired and only the n=1000 tree kept.
ATT_ROOT_FRESH  = MAIN / "outputs" / "paper2" / "section_5_attention_n1000"
PEAKS_CSV       = MAIN / "outputs" / "paper2" / "section_5_attention_n1000" / "_data" / "cross_dataset_peaks.csv"
BBOX_FILE       = MAIN / "inputs" / "irrelevant_number_bboxes.json"

ATT_ROOT_FRESH.mkdir(parents=True, exist_ok=True)
PEAKS_CSV.parent.mkdir(parents=True, exist_ok=True)
assert BBOX_FILE.exists(), f"missing digit-pixel bbox JSON: {BBOX_FILE}"

PDF_OUT = MAIN     / "outputs" / "paper2" / "section_5_figures"
PNG_OUT = WORKTREE / "docs"    / "figures"
PDF_OUT.mkdir(parents=True, exist_ok=True)
PNG_OUT.mkdir(parents=True, exist_ok=True)

GPUS = os.environ.get("VLM_ANCHOR_GPUS", "0,1,2,3,4")  # 5 GPUs by default
RUN_INFERENCE = os.environ.get("VLM_ANCHOR_RUN_INFERENCE", "").lower() in ("1", "true", "yes")  # set VLM_ANCHOR_RUN_INFERENCE=1 to invoke the heavy sharded drivers.

# Pick the attention + e6_steering input roots.
# `ATT_ROOT_FRESH` (section_5_attention_n1000) holds the canonical §5.1
# inference output, so always read from it — the RUN_INFERENCE toggle
# only gates whether a new launch script *adds* to that tree.
ATT_ROOT = ATT_ROOT_FRESH
E6_ROOT  = E6_ROOT_FRESH if RUN_INFERENCE else E6_ROOT_LEGACY
E6_ROOT_FRESH.mkdir(parents=True, exist_ok=True)

print(f"MAIN     = {MAIN}")
print(f"WORKTREE = {WORKTREE}")
print(f"GPUS     = {GPUS}")
print(f"RUN_INFERENCE = {RUN_INFERENCE}")


In [ ]:
def run_cmd(cmd: list[str] | str, *, dry: bool = False, env: dict | None = None) -> int:
    # Print and (optionally) execute a shell command from the main worktree.
    printable = " ".join(shlex.quote(c) for c in cmd) if isinstance(cmd, list) else cmd
    print(f"$ {printable}")
    if dry:
        print("  (dry — RUN_INFERENCE=False)")
        return 0
    full_env = os.environ.copy()
    if env:
        full_env.update(env)
    return subprocess.run(cmd, cwd=MAIN, env=full_env,
                          shell=isinstance(cmd, str)).returncode


def save_figure(fig, stem: str):
    pdf = PDF_OUT / f"{stem}.pdf"
    png = PNG_OUT / f"{stem}.png"
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, bbox_inches="tight", dpi=160)
    print(f"wrote {pdf}")
    print(f"wrote {png}")


## 2 · Recap — what §5.1 + §5.2 established

**§5.1 (per-model peak heterogeneity).** Across the 5-model mechanism
panel the attention peak layer varies from early (Gemma3-4b ≈ L=5 / 42)
to mid (ConvLLaVA, LLaVA-1.5, FastVLM around mid-stack) to late
(OneVision ≈ L=27 / 28 on PlotQA + TallyQA). OneVision additionally
shows a *dataset-dependent* peak shift: InfoVQA pushes the peak back
to L≈14. No uniform causal site.

**§5.2 (within-layer multi-direction).** The K-subspace sweep on
OneVision Main shows a monotonic improvement K=1 → K=2 → K=4 → K=8 in
Δadopt(a) and Δdf(a) at the chosen integration site (L=26, α=1.0). At
L=26 the K=1 fallback fails to clear the anchoring effect across 5
datasets — single direction is insufficient (Figure §5.2b).


In [ ]:
peaks_path = PEAKS_CSV  # isolated reproducer artifact, falls back to legacy below
if not peaks_path.exists():
    legacy = DATA_DIR / "cross_dataset_peaks.csv"
    if legacy.exists():
        print(f"  (fresh CSV missing; reading legacy {legacy} for smoke-only preview)")
        peaks_path = legacy
sweep_path = DATA_DIR / "p4_layer_sweep_per_cell_ci.csv"
pilot_path = DATA_DIR / "e6_pilot_grid_plotqa.csv"

if peaks_path.exists():
    peaks = pd.read_csv(peaks_path)
    print(f"§5.1 peaks rows: {len(peaks)} (source: {peaks_path})")
else:
    print(f"missing: {peaks_path} — run paper_section_5_1_attention_peaks.ipynb first.")
if pilot_path.exists():
    pilot = pd.read_csv(pilot_path)
    print(f"§5.2a pilot cells: {len(pilot)}")
else:
    print(f"missing: {pilot_path} — run paper_section_5_2_subspace_sweep.ipynb first.")
if sweep_path.exists():
    sweep = pd.read_csv(sweep_path)
    print(f"§5.2b sweep cells: {len(sweep)}")
else:
    print(f"missing: {sweep_path} — run paper_section_5_2_subspace_sweep.ipynb first.")


## 3 · Routing-and-integration framework

The two §5 findings co-jointly characterize the anchoring representation
as having two structural properties:

- **Routed across multiple attention layers** — model-specific peak
  layers indicate that different architectures handle anchoring at
  different stages of their forward pass. Uniformity would be the
  exception, not the rule, under this account; the §5.1 panel observes
  exactly that heterogeneity.
- **Integrated into a residual-stream subspace of dimension ≥ 2** —
  the K-monotonic improvement in §5.2 and the K=1 fallback failure
  rule out a single-direction representation. The integration site for
  OneVision Main is `L=26` of 28 layers (≈ 93 % depth), i.e., late in
  the residual stream.

**Mitigation design implications (§6).** The two structural properties
map directly onto the two §6 design choices:
- Choose the *integration site* per architecture (no shared causal site).
- Project out a *multi-direction subspace* (K ≥ 2), not a single
  direction. K=8 captures > 95 % of the explained variance on
  OneVision Main and is the chosen K in the paper's §6.2 mitigation.

**Limitations of §5 evidence.** The peak panel covers 5 architectures
on 3 datasets; further architectures (e.g., Gemma3-27b, Qwen2.5-VL-32b)
would extend the heterogeneity claim. The K-subspace sweep is run
on OneVision Main only; the K-monotonic property is *expected* to be
architecture-general but is *verified* on one model. Cross-architecture
directional verification (γ-β residual-stream bridge on Qwen3-VL) is in
Appendix E and confirms the *direction* of the mitigation but not its
magnitude.


## Summary

§5.3 is interpretation only; it depends on §5.1 + §5.2 outputs being
present on disk. The two §5 findings (peak heterogeneity + within-layer
multi-direction) form the routing-and-integration framework used by §6.

Re-run order:
1. `notebooks/paper_section_5_1_attention_peaks.ipynb` — produces `cross_dataset_peaks.csv`.
2. `notebooks/paper_section_5_2_subspace_sweep.ipynb` — produces `e6_pilot_grid_plotqa.csv` + `p4_layer_sweep_per_cell_ci.csv`.
3. This notebook — reads them back, restates the framework.
